# Model 2: Regional District-Aware Crop Recommendation

This notebook contains the complete end-to-end pipeline for the second model of the research paper. It integrates historical crop production datasets, soil nutrient analysis, and agroclimatic regional data to build a location-aware temporal predictive model.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import xgboost as xgb
import lightgbm as lgb
import shap
import lime.lime_tabular
import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading & Preprocessing

In this section, we merge four diverse raw agricultural datasets to form a robust base table.

In [ ]:
# 1. Base Production Data
base_df = pd.read_csv('../data/raw/crop_production.csv')
base_df.columns = ['State', 'District', 'Year', 'Season', 'Crop', 'Area', 'Production']
base_df['State'] = base_df['State'].astype(str).str.strip().str.upper()
base_df['District'] = base_df['District'].astype(str).str.strip().str.upper()
base_df['Crop'] = base_df['Crop'].astype(str).str.strip().str.upper()
base_df['Season'] = base_df['Season'].astype(str).str.strip().str.upper()

base_df = base_df.dropna(subset=['Production', 'Area'])
base_df = base_df[base_df['Area'] > 0]
base_df['Yield'] = base_df['Production'] / base_df['Area']

# Calculate the dominant crop per State, District, and Season
dominant_crops = base_df.groupby(['State', 'District', 'Season', 'Crop'])['Production'].sum().reset_index()
dominant_crops = dominant_crops.sort_values('Production', ascending=False).drop_duplicates(subset=['State', 'District', 'Season'])
dominant_crops = dominant_crops.drop(columns=['Production'])
dominant_crops['Is_Dominant'] = 1

base_df = base_df.merge(dominant_crops, on=['State', 'District', 'Season', 'Crop'], how='inner')
print(f"Base data shape (Dominant Crops Only): {base_df.shape}")

In [ ]:
# 2. Soil Nutrient Data
soil_df = pd.read_csv('../data/raw/soil-nutrient-analysis.csv')
soil_df['state_name'] = soil_df['state_name'].astype(str).str.strip().str.upper()
soil_df['district_name'] = soil_df['district_name'].astype(str).str.strip().str.upper()
soil_df['nutrient_name'] = soil_df['nutrient_name'].astype(str).str.strip().str.upper()

valid_nutrients = ['NITROGEN', 'PHOSPHORUS', 'POTASSIUM', 'SOIL PH']
soil_filtered = soil_df[soil_df['nutrient_name'].isin(valid_nutrients)]
soil_agg = soil_filtered.groupby(['state_name', 'district_name', 'nutrient_name'])['value'].median().reset_index()
soil_pivot = soil_agg.pivot(index=['state_name', 'district_name'], columns='nutrient_name', values='value').reset_index()
soil_pivot = soil_pivot.rename(columns={'state_name': 'State', 'district_name': 'District', 'PHOSPHORUS': 'PHOSPHOROUS', 'SOIL PH': 'PH'})

In [ ]:
# 3. Rainfall and Climate Data
rain_df = pd.read_csv('../data/raw/district wise rainfall normal.csv')
rain_df['STATE_UT_NAME'] = rain_df['STATE_UT_NAME'].astype(str).str.strip().str.upper()
rain_df['DISTRICT'] = rain_df['DISTRICT'].astype(str).str.strip().str.upper()
rain_df = rain_df.rename(columns={'STATE_UT_NAME': 'State', 'DISTRICT': 'District'})

try:
    agro_gdf = gpd.read_file('../data/raw/Agroclimatic_regions/Agroclimatic_regions.shp')
    agro_gdf['state'] = agro_gdf['state'].astype(str).str.strip().str.upper()
    state_agro = agro_gdf.groupby('state')[['avgtmp_jan', 'avgtmp_jul', 'avgann_rf']].first().reset_index()
    state_agro = state_agro.rename(columns={'state': 'State'})
except:
    print("Warning: Agroclimatic data not loaded.")
    state_agro = pd.DataFrame(columns=['State', 'avgtmp_jan', 'avgtmp_jul', 'avgann_rf'])

In [ ]:
# 4. Merge Data
df = pd.merge(base_df, soil_pivot, on=['State', 'District'], how='left')
df = pd.merge(df, state_agro, on='State', how='left')
df = pd.merge(df, rain_df[['State', 'District', 'ANNUAL', 'Jun-Sep']], on=['State', 'District'], how='left')

for col in ['avgtmp_jan', 'avgtmp_jul', 'avgann_rf']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Merged Dataset Shape: {df.shape}")
df.head()

## 3. Leakage-Free Temporal Feature Engineering

To prevent target leakage, we engineer historical lagged features strictly based on prior years.

In [ ]:
df = df.sort_values(by=['State', 'District', 'Season', 'Crop', 'Year']).reset_index(drop=True)
group_cols = ['State', 'District', 'Season', 'Crop']

# Calculate expanding mean (strictly historic)
df['historical_mean_yield'] = df.groupby(group_cols)['Yield'].apply(lambda x: x.shift(1).expanding().mean()).reset_index(level=group_cols, drop=True)
df['historical_mean_yield'] = df['historical_mean_yield'].fillna(df.groupby(['State', 'Season', 'Crop'])['Yield'].transform('mean'))

# Fill numerical missing values
df = df.fillna(-1)

# Drop crops with too few records
crop_counts = df['Crop'].value_counts()
valid_crops = crop_counts[crop_counts > 500].index
df = df[df['Crop'].isin(valid_crops)]

print(f"Feature engineering complete. Target shape: {df.shape}")

## 4. Modeling Strategy (Train-Test Split)

We split the data into an 80% training set and a 20% unseen test set to accurately evaluate our recommendation models.

In [ ]:
cat_cols = ['State', 'District', 'Season']
num_cols = ['NITROGEN', 'PHOSPHOROUS', 'POTASSIUM', 'PH', 'avgtmp_jan', 'avgtmp_jul', 'avgann_rf', 'ANNUAL', 'Jun-Sep', 'historical_mean_yield']
features = cat_cols + num_cols
target = 'Crop'

# Encode categorical variables
preprocessors = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    preprocessors[col] = le
    
crop_le = LabelEncoder()
df['Crop_Encoded'] = crop_le.fit_transform(df['Crop'].astype(str))

# Random Train-Test Split (80/20) for optimal recommendation accuracy
X = df[features]
y = df['Crop_Encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 5. Model Training & Comparison

We train multiple machine learning models to see which model performs best for regional recommendations.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
    'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1)
}

model_accuracies = {}

print("Training multiple models...")
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    model_accuracies[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")

# Plotting model comparison
plt.figure(figsize=(10, 6))
sns.barplot(x=list(model_accuracies.keys()), y=list(model_accuracies.values()), palette='viridis')
plt.title('Model Accuracy Comparison on Unseen Test Data')
plt.ylabel('Accuracy')
plt.ylim(0, 1.0)
plt.show()

# Selecting best model for further evaluation
best_model_name = max(model_accuracies, key=model_accuracies.get)
print(f"\nBest Model: {best_model_name}")
best_model = models[best_model_name]
preds = best_model.predict(X_test)
probs = best_model.predict_proba(X_test)

## 6. Evaluation & Metrics

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def top_k_accuracy(probs, classes, y_true, k=3):
    correct = 0
    for i, true_class in enumerate(y_true):
        top_k_indices = np.argsort(probs[i])[-k:]
        if true_class in classes[top_k_indices]:
            correct += 1
    return correct / len(y_true)

top1 = accuracy_score(y_test, preds)
top3 = top_k_accuracy(probs, best_model.classes_, y_test.values, k=3)
prec = precision_score(y_test, preds, average='weighted', zero_division=0)
rec = recall_score(y_test, preds, average='weighted', zero_division=0)
f1 = f1_score(y_test, preds, average='weighted', zero_division=0)

print("--- Model Performance on Test Set ---")
print(f"Top-1 Accuracy: {top1:.4f}")
print(f"Top-3 Accuracy: {top3:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}\n")

# Confusion Matrix
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title('Confusion Matrix on Test Data')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 7. Model Explainability (SHAP & LIME)

Explainability is critical for academic publication. Here we utilize SHAP for global feature importance and LIME for local interpretation.

In [ ]:
# SHAP Global Importance
explainer = shap.TreeExplainer(best_model)
X_sample = shap.sample(X_test, 100)
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(10, 6))
# For multiclass, we plot the summary across the first class or average
if len(np.array(shap_values).shape) == 3:
    shap.summary_plot(np.array(shap_values)[:,:,0], X_sample, show=False)
elif isinstance(shap_values, list):
    shap.summary_plot(shap_values[0], X_sample, show=False)
else:
    shap.summary_plot(shap_values, X_sample, show=False)
plt.title('SHAP Global Feature Importance')
plt.show()

In [ ]:
# LIME Local Explanation for a single instance
import lime
import lime.lime_tabular

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=features,
    class_names=[str(c) for c in crop_le.classes_],
    mode='classification'
)

# Pick a random sample from the test set
sample_idx = 0
instance = X_test.iloc[sample_idx]
true_class = crop_le.inverse_transform([y_test.iloc[sample_idx]])[0]

exp = lime_explainer.explain_instance(
    data_row=instance.values,
    predict_fn=best_model.predict_proba,
    num_features=10,
    top_labels=1
)

print(f"LIME Explanation for True Class: {true_class}")
top_label = exp.available_labels()[0]
fig = exp.as_pyplot_figure(label=top_label)
plt.title(f'LIME Local Explanation (True: {true_class}, Predicted Class ID: {top_label})')
plt.tight_layout()
plt.show()

## 8. Making a Region-Wise Crop Recommendation

Finally, we can test our model by providing a specific State, District, and Season, along with local parameters. The model will recommend the top 3 best crops for that region.

In [ ]:
def recommend_crops_for_region(state, district, season, nitrogen, phosphorous, potassium, ph, temp_jan, temp_jul, ann_rf, cur_ann_rf, jun_sep_rf, hist_yield):
    # Encode categorical inputs
    try:
        state_enc = preprocessors['State'].transform([state])[0]
        district_enc = preprocessors['District'].transform([district])[0]
        season_enc = preprocessors['Season'].transform([season])[0]
    except Exception as e:
        print(f"Error encoding input: {e}")
        return
        
    input_features = np.array([[state_enc, district_enc, season_enc, nitrogen, phosphorous, potassium, ph, temp_jan, temp_jul, ann_rf, cur_ann_rf, jun_sep_rf, hist_yield]])
    
    # Predict probabilities using the best model
    probs = best_model.predict_proba(input_features)[0]
    
    # Get top 3 indices
    top_3_idx = np.argsort(probs)[-3:][::-1]
    
    print(f"--- Top 3 Crop Recommendations for {district}, {state} ({season}) ---")
    for i, idx in enumerate(top_3_idx):
        crop = crop_le.inverse_transform([idx])[0]
        probability = probs[idx] * 100
        print(f"{i+1}. {crop} (Confidence: {probability:.2f}%)")

# Let's test with a sample region from our dataset
sample_idx = 10
sample = X_test.iloc[sample_idx]
true_crop = crop_le.inverse_transform([int(y_test.iloc[sample_idx])])[0]

print(f"Testing Region: State={preprocessors['State'].inverse_transform([int(sample['State'])])[0]}, District={preprocessors['District'].inverse_transform([int(sample['District'])])[0]}, Season={preprocessors['Season'].inverse_transform([int(sample['Season'])])[0]}")
print(f"Actual Crop Grown Here: {true_crop}\n")

recommend_crops_for_region(
    state=preprocessors['State'].inverse_transform([int(sample['State'])])[0],
    district=preprocessors['District'].inverse_transform([int(sample['District'])])[0],
    season=preprocessors['Season'].inverse_transform([int(sample['Season'])])[0],
    nitrogen=sample['NITROGEN'],
    phosphorous=sample['PHOSPHOROUS'],
    potassium=sample['POTASSIUM'],
    ph=sample['PH'],
    temp_jan=sample['avgtmp_jan'],
    temp_jul=sample['avgtmp_jul'],
    ann_rf=sample['avgann_rf'],
    cur_ann_rf=sample['ANNUAL'],
    jun_sep_rf=sample['Jun-Sep'],
    hist_yield=sample['historical_mean_yield']
)